# Loading script for Clean&Safe hotels in DWHPFX schema

In [1]:
import requests
import json
import pandas as pd
import time
import configparser
import pyexasol

C:\Anaconda\lib\site-packages\numpy\_distributor_init.py:32: UserWarning: loaded more than 1 DLL from .libs:
C:\Anaconda\lib\site-packages\numpy\.libs\libopenblas.QVLO2T66WEPI7JZ63PS3HMOHFEY472BC.gfortran-win_amd64.dll
C:\Anaconda\lib\site-packages\numpy\.libs\libopenblas.TXA6YQSD3GCQQC22GEQ54J2UDCXDXHWN.gfortran-win_amd64.dll
  stacklevel=1)


In [7]:
def get_token(urlLogin, env):
    """ This function authenticates and returns token and refreshToken """
    auth = {"client_id": "f6eeddfe-b600-4fc8-b283-e66f5fa0e0aa", "client_secret": "45f8160f-0bee-43af-a7ac-3d1013c84a78"}
    response = requests.post(urlLogin, json=auth)
    if response.status_code == 200:
        data = response.text
        parsed = json.loads(data)
        token = parsed['token']
        refreshToken = parsed['refreshToken']
    return token, refreshToken

def cleanNsafeExtract(urlAuth, env='det'):
    """ This function fetches the data from the api and stores in a df """
    token, refreshToken = get_token(urlAuth,env)
    startTime = time.time()
    ### API get link
    urlGet = f"https://api.hotel-audit.hrs.com/v2/audits/report?token={token}&page=1&size=500"
    getResponse = requests.get(urlGet)
    
    if getResponse.status_code == 200:
        data = getResponse.text
        parsed = json.loads(data)
        print(f"Status code: {getResponse.status_code}")
    else:
        print("Error in fetching the pages")
        
    try:
       ### Initialize the dataframe
        dfRaw = pd.DataFrame()
        df = pd.DataFrame()
        miss_dict = {              
          '1':'Public health authorities guidelines'
        , '2':'Risk assessment'
        , '3':'Risk assessment - hotstaff'
        , '4':'Action plan'
        , '5':'Log books'
        , '6':'Staff/Guests well informed'
        , '7':'Staff/Guests assessed'
        , '8':'Pictograms'
        , '9':'Updated contact info'
        , '10':'Staff training'
        , '11':'Tracing system'
        , '12':'Hygiene - suppliers/contractors'
        , '13':'Web checkin'
        , '14':'Hothygiene - guests'
        , '15':'Common area- social distancing'
        , '16':'PPE/kits - staff'
        , '17':'PPE/kits - guests'
        , '18':'Staff remind guests'
        , '19':'Disinfectant - room keys'
        , '20':'Disinfected pools'
        , '21':'Lockers - Spa/fitness social distancing'
        , '22':'Lockers - Spa/fitness'
        , '23':'Disinfectant products - fitness'
        , '24':'Disinfectant reminder - fitness'
        , '25':'Fitness equipment'
        , '26':'Social distancing - Spa/fitness'
        , '27':'Dish washers and washing machine'
        , '28':'HVAC systems'
        , '29':'Sanitiser dispensers'
        , '30':'Public restrooms'
        , '31':'Take away/Room service'
        , '32':'Safety measures - buffets'
        , '33':'Disinfection - buffet areas'
        , '34':'Disinfection - vending machines'
        , '35':'Dishwashing machine'
        , '36':'Social distancing dining'
        , '37':'Social distancing seating'
        , '38':'Guest reminder'
        , '39':'Cleaning protocol - public areas'
        , '40':'Disinfection protocol - COVID cases'
        , '41':'Disinfection - guest rooms'
        , '42':'Dirty linen storage'
        , '43':'Ventilation'
        , '44':'Staff to access meeting room'
        , '45':'Luggage store for group'
        , '46':'Social distancing - meeting rooms'
        , '':''
        }
        
        for x in range(parsed['total_pages']):
            url_iter = f"https://api.hotel-audit.hrs.com/v2/audits/report?token={token}&page={x+1}&size=500"
            response = requests.get(url_iter)
            if response.status_code == 200 or response.status_code == 403:    
                data = response.text
                parsed = json.loads(data)
                dfTemp = pd.DataFrame(parsed.get('results'))
                dfRaw = dfRaw.append(dfTemp, ignore_index=True)
            else: 
                print("Request failed page: {} ".format(x))
        dfRaw['created_date']= dfRaw['created_date'].astype(str).str[:-6]
        dfRaw['updated_date']= dfRaw['updated_date'].astype(str).str[:-6]
        dfRaw['audit_date']= dfRaw['audit_date'].astype(str).str[:-6]
        dfRaw['created_date']= pd.to_datetime(dfRaw['created_date'], utc=False)
        dfRaw['updated_date']= pd.to_datetime(dfRaw['updated_date'], utc=False)
        dfRaw['audit_date']= pd.to_datetime(dfRaw['audit_date'], utc=False)
        df = dfRaw[dfRaw.hkey.notnull()]
        df = df.reset_index()

        # Removing duplicates in missed column
        for index, row in df.iterrows():
            if row.missed != '':
                string = ",".join(str(x).strip() for x in list(set(row['missed'].split(','))))
                df.loc[index, 'missed'] = string
            else:
                df.loc[index, 'missed'] = ''

        # Creating a description column for the missed values
        missed_desc = []
        for row in df.itertuples(name='missed'): 
            temp = row.missed.split(',')
            missed_desc.append([miss_dict[temp[i]] for i in range(len(temp))])
        temp = [str(item) for item in missed_desc]
        missed_df = pd.DataFrame(temp)
        df = df.join(missed_df)
        df.rename(columns={0: "missed_desc"}, inplace = True)
        df['missed_desc'][df.missed_desc.str.len() < 5] = None
        df.drop(['index'], axis= 1, inplace = True)
        df['LDTS'] = time.strftime('%Y-%m-%d')
        endTime = time.time() - startTime
        print(f'Number of records having HOTEL_IDs: {dfRaw.id[dfRaw.hkey.notnull()].count()}.')
        print(f'Number of records with no HOTEL_IDs: {dfRaw.id[dfRaw.hkey.isna()].count()}.')
        print(f'Time in minutes: {round(endTime/60,2)}')
    except Exception as e:
        endTime = time.time() - startTime
        print('Failed in function cleanNsafe - ')
        print(f'Time in minutes: {round(endTime/60,2)}')
        raise e
    return df


def cleanNsafeLoad(df): 
    """ This function is to load the data into Exasol DB -> DWHPFX schema """
    try:
        #Location of the ini file
        config = configparser.ConfigParser()
        ## Config location CHANGE
        config.read('C:\\Users\\svi02\\.spyder-py3\\pfxDET.ini')
        dsn=config['pfxDET']['dsn']
        user=config['pfxDET']['user']
        pwd=config['pfxDET']['pwd']
        schema=config['pfxDET']['schema']
        # Exasol connection
        connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
        connect.execute("TRUNCATE TABLE DWHPFX.CLEAN_SAFE_HOTELS")
        connect.import_from_pandas(df, table = ('DWHPFX','CLEAN_SAFE_HOTELS'))
        stmt = connect.last_statement()
        print(f'Number of records inserted  {stmt.rowcount()}.')
    except Exception as e:
        print('Failed in function GreenStayLoad - ')
        raise e

### Function call

In [8]:
df = cleanNsafeExtract('https://api.hotel-audit.hrs.com/auth/login')
cleanNsafeLoad(df)

Number of records inserted  65934.


### Test results in excel

In [106]:
from datetime import date
df.to_excel('C:\\Users\\svi02\\Documents\\misc\\'+str(date.today())+'_test.xlsx', 
              sheet_name='Sheet1', 
              header=True,
              encoding='utf-8',
              index=False,
              freeze_panes=(1,0) )